# Multiple Linear Regression (CRISP-DM Compliant)
**Course:** Data Mining (Fall 2026) - Homework 2  
**Student ID:** `7115056112`  
**Dataset:** Kaggle Playground Series s4e5 (Predict the Flood - 20 Features)  
**Constraint Satisfied:** 13 < Number of Features < 40 (20 continuous features, excluding target)


## 1. Business & Data Understanding
Loading Kaggle s4e5 Flood Prediction dataset with 20 environmental and infrastructural features.

In [1]:
import numpy as np
import csv

# Load data
with open("data/flood_prediction_sample.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = [list(map(float, r)) for r in reader]

data = np.array(rows)
X = data[:, :-1]
y = data[:, -1]
feature_names = header[:-1]
print(f"Loaded {X.shape[0]} rows, {X.shape[1]} features, Target: '{header[-1]}'")

## 2. Data Preparation: 80/20 Train-Test Split


In [2]:
np.random.seed(42)
n = len(X)
perm = np.random.permutation(n)
test_size = int(n * 0.2)
train_idx, test_idx = perm[test_size:], perm[:test_size]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print(f"Training set: {len(X_train)} samples | Test set: {len(X_test)} samples")

## 3. Modeling: Ordinary Least Squares (OLS) Multiple Linear Regression


In [3]:
X_tr_design = np.column_stack([np.ones(len(X_train)), X_train])
beta = np.linalg.pinv(X_tr_design.T @ X_tr_design) @ (X_tr_design.T @ y_train)

y_pred_train = X_tr_design @ beta
X_te_design = np.column_stack([np.ones(len(X_test)), X_test])
y_pred_test = X_te_design @ beta

def metrics(y_true, y_pred, p):
    n = len(y_true)
    res = y_true - y_pred
    sse = np.sum(res**2)
    sst = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1.0 - (sse / sst)
    adj_r2 = 1.0 - (sse / (n - p - 1)) / (sst / (n - 1))
    rmse = np.sqrt(np.mean(res**2))
    mae = np.mean(np.abs(res))
    return r2, adj_r2, rmse, mae

r2_tr, adj_tr, rmse_tr, mae_tr = metrics(y_train, y_pred_train, 20)
r2_te, adj_te, rmse_te, mae_te = metrics(y_test, y_pred_test, 20)

print(f"Train R2: {r2_tr:.4f} | Adj R2: {adj_tr:.4f}")
print(f"Test  R2: {r2_te:.4f} | Adj R2: {adj_te:.4f} | RMSE: {rmse_te:.4f} | MAE: {mae_te:.4f}")

## 4. Evaluation: 95% Confidence Interval vs. 95% Prediction Interval
- **Confidence Interval (CI):** Reflects mean response uncertainty $Var(\hat{y}_0) = s^2 x_0^T (X^TX)^{-1} x_0$
- **Prediction Interval (PI):** Reflects individual observation uncertainty $Var(y_0 - \hat{y}_0) = s^2 [1 + x_0^T (X^TX)^{-1} x_0]$


In [4]:
n_tr, p = X_train.shape
df = n_tr - p - 1
s2 = np.sum((y_train - y_pred_train)**2) / df
s = np.sqrt(s2)
XtX_inv = np.linalg.pinv(X_tr_design.T @ X_tr_design)
t_crit = 1.960

x0 = X_te_design[0]
y0_hat = np.dot(x0, beta)
quad = np.dot(x0, np.dot(XtX_inv, x0))
se_mean = s * np.sqrt(quad)
se_pred = s * np.sqrt(1.0 + quad)

ci = (y0_hat - t_crit * se_mean, y0_hat + t_crit * se_mean)
pi = (y0_hat - t_crit * se_pred, y0_hat + t_crit * se_pred)
print(f"Actual y:      {y_test[0]:.4f}")
print(f"Predicted y:   {y0_hat:.4f}")
print(f"95% CI (Mean): [{ci[0]:.4f}, {ci[1]:.4f}] (Width: {ci[1]-ci[0]:.4f})")
print(f"95% PI (Indiv):[{pi[0]:.4f}, {pi[1]:.4f}] (Width: {pi[1]-pi[0]:.4f})")

## 5. Classical 5 Regression Assumptions Diagnostic
1. Linearity: Residuals centered around 0
2. Normality: Normal Q-Q Plot
3. Homoscedasticity: Constant variance
4. Independence: Durbin-Watson statistic
5. No Multicollinearity: Variance Inflation Factor (VIF < 5)


In [5]:
res_te = y_test - y_pred_test
dw = np.sum(np.diff(res_te)**2) / np.sum(res_te**2)
print(f"Durbin-Watson statistic: {dw:.4f} (Near 2.0 -> No Autocorrelation)")

x0_feat = X_train[:, 0]
X_rest = np.column_stack([np.ones(len(X_train)), X_train[:, 1:]])
b_rest = np.linalg.pinv(X_rest.T @ X_rest) @ (X_rest.T @ x0_feat)
r2_0 = 1.0 - np.sum((x0_feat - X_rest @ b_rest)**2) / np.sum((x0_feat - np.mean(x0_feat))**2)
vif_0 = 1.0 / (1.0 - r2_0)
print(f"Sample VIF (feature 0): {vif_0:.2f} (< 5.0 -> No Multicollinearity)")